In [1]:
import numpy as np
import pandas as pd
import torch
import math
import sqlite3
from   ucimlrepo import fetch_ucirepo
from   sklearn.model_selection import train_test_split
from   itertools import chain, combinations
from   more_itertools import powerset


path = 'wine.csv'
wine = pd.read_csv(path)



In [2]:
X = wine.drop(columns=['quality'])
y = wine['quality']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

players = list(X.columns)
target  = 'quality'

X_cov     = np.array(X_train.cov())
X_cov_inv = np.linalg.inv(X_cov)

X_train, y_train = np.array(X_train), np.array(y_train)
X_test,  y_test  = np.array(X_test),  np.array(y_test)


In [3]:
# Cell used to work out syntax of value function

start_idx, end_idx       = 1, 7
start_point,  end_point   = X_test[start_idx,:], X_test[end_idx,:]
start_target, end_target  = y_test[start_idx],   y_test[end_idx]
X_explain                = np.vstack([X_train, start_point.T, end_point.T])
y_explain                = y_train + start_target + end_target

diff = X_explain - start_point
md   = np.sqrt(np.einsum('ij,jk,ik->i', diff, X_cov_inv, diff))
print(md[np.argmin(md, axis=0)], md[np.argmax(md, axis=0)])
print(np.argmin(md, axis=0), np.argmax(md, axis=0))




0.0 39.767807626712255
2192 2222


In [21]:
def value2(coalition, start_idx, end_idx,  X_train, y_train):

    vertex = X_test[start_idx]
    if len(coalition) > 0:
        for player in coalition:
            vertex[player] = X_test[end_idx][player]
    X_explain                = np.vstack([X_train, X_test[start_idx].T, X_test[end_idx]])
    y_explain                = np.append(y_train,[y_test[start_idx], y_test[end_idx]])

    print(vertex)

    diff = X_explain - vertex
    md   = np.sqrt(np.einsum('ij,jk,ik->i', diff, X_cov_inv, diff))

    # Find the target value of the closest point
    nearest = y_explain[np.argmin(md, axis=0)]

    return nearest


print(value2({0}, 1,9,X_train,y_train))


[8.70000e+00 3.10000e-01 7.30000e-01 1.43500e+01 4.40000e-02 2.70000e+01
 1.91000e+02 1.00013e+00 2.96000e+00 8.80000e-01 8.70000e+00]
5


In [178]:
def value(coalition, start_idx, end_idx,  X_train, y_train):

    # Add the two contrastive data points to the set of explaining data
    start_point,  end_point  = X_test[start_idx], X_test[end_idx]
  #  print(start_idx, end_idx)
    start_target, end_target = y_test[start_idx], y_test[end_idx]
  #  print('start', start_target, start_point)
  #  print('end', end_target, end_point)
    start_target, end_target = y_test[start_idx],   y_test[end_idx]
  #  print(start_target, end_target)
    X_explain                = np.vstack([X_train, start_point.T, end_point.T])
    y_explain                = np.append(y_train,[start_target, end_target])

 #   print('start', start_point)
 #   print('end', end_point)

    # Use the player indicies to build the coordinates of the vertex
    vertex = start_point[:]
  #  print('start_vertex', vertex)
   # print(type(vertex), len(coalition))
    if len(coalition) > 0:
   #     print('here', coalition)
        for player in coalition:
    #        print('and here')
            vertex[player] = end_point[player]
    #print('end_vertex', vertex)

    # Calculate the Mahalanobis distance of each item in the X_explain data
    diff = X_explain - vertex
    md   = np.sqrt(np.einsum('ij,jk,ik->i', diff, X_cov_inv, diff))

    # Find the target value of the closest point
    nearest = y_explain[np.argmin(md, axis=0)]

    return nearest

for S in powerset({0,1,2,3,4,5,6,7,8,9,10}):
    print('Function Test: ',S,  value(S, 2, 7,  X_train, y_train))

Function Test:  () 8
Function Test:  (0,) 8
Function Test:  (1,) 8
Function Test:  (2,) 8
Function Test:  (3,) 8
Function Test:  (4,) 8
Function Test:  (5,) 8
Function Test:  (6,) 8
Function Test:  (7,) 8
Function Test:  (8,) 8
Function Test:  (9,) 8
Function Test:  (10,) 8
Function Test:  (0, 1) 8
Function Test:  (0, 2) 8
Function Test:  (0, 3) 8
Function Test:  (0, 4) 8
Function Test:  (0, 5) 8
Function Test:  (0, 6) 8
Function Test:  (0, 7) 8
Function Test:  (0, 8) 8
Function Test:  (0, 9) 8
Function Test:  (0, 10) 8
Function Test:  (1, 2) 8
Function Test:  (1, 3) 8
Function Test:  (1, 4) 8
Function Test:  (1, 5) 8
Function Test:  (1, 6) 8
Function Test:  (1, 7) 8
Function Test:  (1, 8) 8
Function Test:  (1, 9) 8
Function Test:  (1, 10) 8
Function Test:  (2, 3) 8
Function Test:  (2, 4) 8
Function Test:  (2, 5) 8
Function Test:  (2, 6) 8
Function Test:  (2, 7) 8
Function Test:  (2, 8) 8
Function Test:  (2, 9) 8
Function Test:  (2, 10) 8
Function Test:  (3, 4) 8
Function Test:  (3, 5)

In [173]:
X_test[7]

array([7.60e+00, 2.60e-01, 3.60e-01, 1.60e+00, 3.20e-02, 6.00e+00,
       1.06e+02, 9.93e-01, 3.15e+00, 4.00e-01, 1.04e+01])

In [133]:
def explain(start_idx, end_idx, X_train, y_train):

    def gamma(players, coalition):
        N = len(players)
        S = len(coalition)
        return math.factorial(S) * math.factorial(N - S - 1) / math.factorial(N)

    phi = dict()

    player_set = {0,1,2,3,4,5,6,7,8,9,10}

    for player in player_set:
        coalitions = player_set - {player}
        phi[player]=0.0
        for S in powerset(coalitions):
            v2 = value(set(S).union({player}), start_idx, end_idx,  X_train, y_train)
            v1 = value(set(S), start_idx, end_idx,  X_train, y_train)
            if v2 != 0: print(player, v2, v1)
            phi[player] += gamma(player_set, S) * (v2 - v1)

        #print(player, X_test[start_idx, player], X_test[end_idx, player], round(phi[player],3))



    return y_train[start_idx], y_train[end_idx], phi

start_val, end_val, phi = explain(1, 7, X_train, y_train)


0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 8
0 8 

In [130]:
print(start_val, end_val)
for player, attribution in phi.items():
    print(player, attribution)

5 6
0 0.0
1 0.0
2 0.0
3 0.0
4 0.0
5 0.0
6 0.0
7 0.0
8 0.0
9 0.0
10 0.0


In [81]:
player_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid','residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']

target_player = 'quality'

player_set = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}

explain()






In [ ]:
conn.close()

In [ ]:
playr